# 04 - Data Cleaning

**Input:** `data/interim/train.csv` | `val.csv` | `test.csv`

**Output:** `data/processed/train_clean.csv` | `val_clean.csv` | `test_clean.csv`

**Artifacts:** `artifacts/cleaning_artifacts.json` · `lof_model.pkl` · `lof_scaler.pkl`

**Config:** `configs/data_config.yaml` → `eda_derived` section (missingness, lof)

---

### What this notebook does

| Step | Action | Source |
|---|---|---|
| 1 | **Strict median imputation** on declared columns | `eda_derived.missingness` (config) |
| 2 | **LOF outlier flag** (fit on train, applied to all splits) | `eda_derived.lof` (config) |
| 3 | **Save train-fit statistics** as serialized artifacts | `artifacts/` |

### FIT/TRANSFORM rule

All statistics (imputation median, LOF model, StandardScaler) are **fit on train only** and applied to val/test. This prevents data leakage from test information into training decisions.

> **Note on omitted transformations:**
> - `is_capped` is **intentionally excluded** to prevent target leakage (it is derived from `median_house_value`).
> - `log1p` is **deferred** to `engineering.py` to avoid corrupting ratio features (e.g., `rooms / households`).

---
## 0 - Setup & Load

In [8]:
# =============================================================================
# 0. SETUP & LOAD
# =============================================================================

import os
import sys
from pathlib import Path

# Step 1: Add the well-known Colab path temporarily to allow importing src
_colab_repo_path = Path("/content/california_housing_full_project")
if _colab_repo_path.exists() and str(_colab_repo_path) not in sys.path:
    sys.path.insert(0, str(_colab_repo_path))

# Step 2: Import the official project path from the centralized module
from src.utils.paths import PROJECT_DIR

# Step 3: Override repo_path with the official path, change directory, and re-add to sys.path
repo_path = PROJECT_DIR
os.chdir(repo_path)

if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

print(f"✅ Project root : {PROJECT_DIR}")
print(f"✅ Working dir  : {os.getcwd()}")
print(f"✅ sys.path[0]  : {sys.path[0]}")

✅ Project root : /content/california_housing_full_project
✅ Working dir  : /content/california_housing_full_project
✅ sys.path[0]  : /content/california_housing_full_project


---
## 1 - Imports

In [17]:
import logging

from src.data.cleaning import (
    CleaningError,
    load_eda_config,
    run_cleaning,
)
from src.data.data_loader import DataLoader
from src.utils.logger import get_logger, setup_logging

setup_logging(level=logging.INFO)
logger = get_logger("notebook.04_cleaning")

CONFIG_PATH = repo_path / "configs" / "data_config.yaml"

print("Imports ready")

Imports ready


---
## 2 - Load Interim Splits

In [3]:
loader = DataLoader()

train = loader.load_interim("train.csv")
val   = loader.load_interim("val.csv")
test  = loader.load_interim("test.csv")

print(f"train : {train.shape[0]:,} rows x {train.shape[1]} cols")
print(f"val   : {val.shape[0]:,} rows x {val.shape[1]} cols")
print(f"test  : {test.shape[0]:,} rows x {test.shape[1]} cols")
print()
print("Null counts (train):")
nulls = train.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.sum() > 0 else "  No nulls")

2026-08-30 01:31:05 | INFO     | src.data.data_loader | DataLoader initialized | Drive mode: True
2026-08-30 01:31:05 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/interim/train.csv
2026-08-30 01:31:05 | INFO     | src.data.data_loader | Loaded 'train.csv' | shape=(14448, 10) | stage=interim
2026-08-30 01:31:05 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/interim/val.csv
2026-08-30 01:31:05 | INFO     | src.data.data_loader | Loaded 'val.csv' | shape=(3096, 10) | stage=interim
2026-08-30 01:31:05 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/interim/test.csv
2026-08-30 01:31:05 | INFO     | src.data.data_loader | Loaded 'test.csv' | shape=(3096, 10) | stage=interim
train : 14,448 rows x 10 cols
val   : 3,096 rows x 10 cols
test  : 3,096 rows x 10 cols

Null counts (train):
total_bedrooms    140


---
## 3 - Verify EDA Config

Before cleaning, confirm that `data_config.yaml` has the `eda_derived` section
with real values from `notebooks/03_eda.ipynb`. If `source = fallback`, stop and
update the config first.

In [10]:
eda_cfg = load_eda_config(CONFIG_PATH)

print(f"EDA config source    : {eda_cfg.source if hasattr(eda_cfg, 'source') else 'config'}")
print(f"Imputation columns   : {eda_cfg.impute_columns}")
print(f"Imputation strategy  : {eda_cfg.imputation_strategy}")
print(f"LOF features         : {eda_cfg.lof_features}")
print(f"LOF contamination    : {eda_cfg.lof_contamination}")
print(f"LOF n_neighbors      : {eda_cfg.lof_n_neighbors}")
print()

if not eda_cfg.impute_columns:
    print("⚠️  WARNING: No columns declared for imputation.")
    print("Check eda_derived.missingness[].impute: true in data_config.yaml")
else:
    print("✅ Config loaded from data_config.yaml - ready to clean")

2026-08-30 01:36:48 | INFO     | src.data.cleaning | EDA configuration loaded successfully.
2026-08-30 01:36:48 | INFO     | src.data.cleaning | Imputation columns: ['total_bedrooms']
2026-08-30 01:36:48 | INFO     | src.data.cleaning | LOF features: ['median_income', 'total_rooms', 'population', 'households', 'longitude', 'latitude']
2026-08-30 01:36:48 | INFO     | src.data.cleaning | LOF contamination: 0.02
2026-08-30 01:36:48 | INFO     | src.data.cleaning | LOF n_neighbors: 20
EDA config source    : config
Imputation columns   : ['total_bedrooms']
Imputation strategy  : {'total_bedrooms': 'median'}
LOF features         : ['median_income', 'total_rooms', 'population', 'households', 'longitude', 'latitude']
LOF contamination    : 0.02
LOF n_neighbors      : 20

✅ Config loaded from data_config.yaml - ready to clean


---
## 4 - Run Cleaning Pipeline

`run_cleaning()` executes the full cleaning stage:

1. Fit imputer on train → apply to all splits
2. Fit LOF (with StandardScaler) on train → flag outliers in all splits
3. Save train-fit artifacts: `cleaning_artifacts.json`, `lof_model.pkl`, `lof_scaler.pkl`
4. Save cleaned datasets to `data/processed/`

> **Note:** `is_capped` and `log1p` are not part of cleaning — they are excluded/deferred to prevent leakage and preserve ratio features. DVC tracking is handled externally by `dvc.yaml`.

In [12]:
try:
    result = run_cleaning(
    train=train,
    val=val,
    test=test,
    config_path=CONFIG_PATH,
    save_artifacts_flag=True,
    )
    
    print(result.summary())
except CleaningError as e:
    logger.error(f"❌ Cleaning pipeline failed: {e}")
    print(f"ERROR: Cleaning failed - {e}")
    raise
except Exception as e:
    logger.error(f"❌ Unexpected error during cleaning: {e}")
    print(f"UNEXPECTED ERROR: {e}")
    raise

2026-08-30 01:38:08 | INFO     | src.data.cleaning | ======================================================================
2026-08-30 01:38:08 | INFO     | src.data.cleaning | DATA CLEANING STARTED
2026-08-30 01:38:08 | INFO     | src.data.cleaning | ======================================================================
2026-08-30 01:38:08 | INFO     | src.data.cleaning | Step 1/6 - Validating cleaning inputs...
2026-08-30 01:38:08 | INFO     | src.data.cleaning | Step 2/6 - Loading EDA-derived configuration...
2026-08-30 01:38:08 | INFO     | src.data.cleaning | EDA configuration loaded successfully.
2026-08-30 01:38:08 | INFO     | src.data.cleaning | Imputation columns: ['total_bedrooms']
2026-08-30 01:38:08 | INFO     | src.data.cleaning | LOF features: ['median_income', 'total_rooms', 'population', 'households', 'longitude', 'latitude']
2026-08-30 01:38:08 | INFO     | src.data.cleaning | LOF contamination: 0.02
2026-08-30 01:38:08 | INFO     | src.data.cleaning | LOF n_neighbors

2026-08-30 01:38:09 | INFO     | src.data.cleaning | LOF fitted successfully.
2026-08-30 01:38:09 | INFO     | src.data.cleaning | Rows: 14,448
2026-08-30 01:38:09 | INFO     | src.data.cleaning | Features: ['median_income', 'total_rooms', 'population', 'households', 'longitude', 'latitude']
2026-08-30 01:38:09 | INFO     | src.data.cleaning | Contamination: 0.02
2026-08-30 01:38:09 | INFO     | src.data.cleaning | n_neighbors: 20
2026-08-30 01:38:09 | INFO     | src.data.cleaning | Step 5/6 - Applying train-fitted LOF...
2026-08-30 01:38:09 | INFO     | src.data.cleaning | [train] LOF outliers: 240 / 14,448
2026-08-30 01:38:09 | INFO     | src.data.cleaning | [val] LOF outliers: 49 / 3,096
2026-08-30 01:38:09 | INFO     | src.data.cleaning | [test] LOF outliers: 62 / 3,096
2026-08-30 01:38:09 | INFO     | src.data.cleaning | Step 6/6 - Saving outputs and artifacts...
2026-08-30 01:38:09 | INFO     | src.data.cleaning | Cleaning artifacts saved to: /content/california_housing_full_proj

---
## 5 - Verify Cleaning Results

Four checks after cleaning:

- **Nulls removed** → all splits should have zero missing values
- **`lof_outlier` present** → outlier flag column must exist in all splits
- **`is_capped` absent** → target‑derived column must NOT leak into cleaned data
- **`lof_outlier` distribution** → verify outlier counts are reasonable

In [13]:
print("-- Null check after cleaning --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    nulls = df.isnull().sum().sum()
    status = "OK" if nulls == 0 else f"FAIL ({nulls} nulls remaining)"
    print(f"  {name:<6} : {status}")

-- Null check after cleaning --
  train  : OK
  val    : OK
  test   : OK


In [14]:
print("\n-- New columns check --")
# 1. lof_outlier MUST exist
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    present = "lof_outlier" in df.columns
    print(f"  lof_outlier     in {name:<6} : {'OK' if present else 'MISSING'}")

# 2. is_capped MUST NOT exist (target leakage prevention)
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    present = "is_capped" in df.columns
    status = "OK (blocked)" if not present else "FAIL (leakage!)"
    print(f"  is_capped       in {name:<6} : {status}")

print("\n-- lof_outlier distribution --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    counts = df["lof_outlier"].value_counts().sort_index()
    print(f"  {name:<6} : {dict(counts)} (0=inlier, 1=outlier)")


-- New columns check --
  lof_outlier     in train  : OK
  lof_outlier     in val    : OK
  lof_outlier     in test   : OK
  is_capped       in train  : OK (blocked)
  is_capped       in val    : OK (blocked)
  is_capped       in test   : OK (blocked)

-- lof_outlier distribution --
  train  : {0: np.int64(14208), 1: np.int64(240)} (0=inlier, 1=outlier)
  val    : {0: np.int64(3047), 1: np.int64(49)} (0=inlier, 1=outlier)
  test   : {0: np.int64(3034), 1: np.int64(62)} (0=inlier, 1=outlier)


---
## 6 - Shape & Schema Consistency

In [16]:
print("-- Shape check --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    print(f"  {name:<6} : {df.shape[0]:,} rows x {df.shape[1]} cols")

print()
print("-- Column consistency --")
train_cols = list(result.train.columns)
for name, df in [("val", result.val), ("test", result.test)]:
    if list(df.columns) == train_cols:
        print(f"  {name} columns match train")
    else:
        diff = set(train_cols) ^ set(df.columns)
        print(f"  {name} column mismatch: {diff}")

print()
print("-- Final column list --")
print(result.train.columns.tolist())

-- Shape check --
  train  : 14,448 rows x 11 cols
  val    : 3,096 rows x 11 cols
  test   : 3,096 rows x 11 cols

-- Column consistency --
  val columns match train
  test columns match train

-- Final column list --
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity', 'lof_outlier']


---
## Summary & Next Steps

| Done | Details |
|---|---|
| Nulls imputed | `total_bedrooms` median imputation (declared in config) |
| LOF outlier flag | contamination=0.02, novelty=True, fitted on train only |
| `median_income` | unchanged (no transform applied) |
| Files saved | `data/processed/` (`train_clean.csv`, `val_clean.csv`, `test_clean.csv`) |
| Artifacts saved | `artifacts/cleaning_artifacts.json` + `lof_model.pkl` + `lof_scaler.pkl` |
| Target leakage prevention | ✅ `is_capped` excluded from cleaning |
